In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import plotly.graph_objects as go
#import plotly.express as px
from plotly_resampler import FigureResampler, register_plotly_resampler
from datetime import datetime
from datetime import timedelta
from sklearn.preprocessing import MinMaxScaler
import random
scaler = MinMaxScaler()

In [ ]:
def generate_sincurve(year, amplitude=1.0, frequency=1.0):
    start_date = datetime(year, 1, 1, 0, 0)
    end_date = datetime(year + 1, 1, 1, 0, 0)
    total_minutes = int((end_date - start_date).total_seconds() / 24)
    timestamps = [start_date + timedelta(minutes=i) for i in range(total_minutes)]
    sin_values = amplitude * np.sin(2 * np.pi * frequency * np.arange(total_minutes) / (24 * 60))

    df = pd.DataFrame({
        'tampstamp': timestamps,
        'values': sin_values
    })
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    return df

def generate_sincurve_with_anomalies(year, amplitude=1.0, frequency=1.0, anomaly_fraction=0.0001, anomaly_magnitude=7.0):
    start_date = datetime(year, 1, 1, 0, 0)
    end_date = datetime(year + 1, 1, 1, 0, 0)
    total_minutes = int((end_date - start_date).total_seconds() / 12)
    timestamps = [start_date + timedelta(minutes=i) for i in range(total_minutes)]
    sin_values = amplitude * np.sin(2 * np.pi * frequency * np.arange(total_minutes) / (24 * 60))

    # Introduce anomalies
    num_anomalies = int(total_minutes * anomaly_fraction)
    anomaly_indices = random.sample(range(total_minutes), num_anomalies)
    for idx in anomaly_indices:
        sin_values[idx] += anomaly_magnitude * (random.random() - 0.5) * 2  # Randomly add/subtract anomaly_magnitude

    df = pd.DataFrame({
        'timestamp': timestamps,
        'values': sin_values
    })
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    return df

In [27]:

tmp = generate_sincurve_with_anomalies(2023)
print(len(tmp))
time_series_data = tmp['values'].values
time_series_data_normalized = scaler.fit_transform(time_series_data.reshape(-1, 1))

def create_dataset(data, time_step=1):
    X = []
    for i in range(len(data) - time_step):
        X.append(data[i:(i + time_step), 0])
    return np.array(X)

time_step = 10
X = create_dataset(time_series_data_normalized, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

2628000


In [28]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64, input_shape=(time_step, 1), return_sequences=True),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.RepeatVector(time_step),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1))
])

model.compile(optimizer='adam', loss='mse')
model.summary()

# Train the model
history = model.fit(X, X, epochs=1, batch_size=32, validation_split=0.2, shuffle=False)

X_pred = model.predict(X)
mse = np.mean(np.power(X.reshape(X.shape[0], time_step) - X_pred.reshape(X_pred.shape[0], time_step), 2), axis=1)

# Threshold for anomaly detection
threshold = np.percentile(mse, 95)

c:\Users\jacks\Documents\yaf-docker\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                  │ (None, 10, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_3 (RepeatVector)  │ (None, 10, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, 10, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 10, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 10, 1)          │            65 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,529 (244.25 KB)

 Trainable params: 62,529 (244.25 KB)

 Non-trainable params: 0 (0.00 B)

65700/65700 ━━━━━━━━━━━━━━━━━━━━ 655s 10ms/step - loss: 0.0018 - val_loss: 2.1691e-04
82125/82125 ━━━━━━━━━━━━━━━━━━━━ 257s 3ms/step


In [29]:
from plotly_resampler import FigureWidgetResampler


anomalies = mse > threshold
anomalies_index = np.where(anomalies)[0]

print(f'Number of anomalies detected: {len(anomalies_index)}')
print(f'Indices of anomalies: {anomalies_index}')
fig = go.Figure()
register_plotly_resampler(mode="auto", default_n_shown_samples=1500)

# Plot anomalies
fig.add_trace(go.Scatter(
    x=tmp['timestamp'],
    y=tmp['values'],
    mode='lines',
    name='Time Series Data'
))

# Add the anomalies
fig.add_trace(go.Scatter(
    x=tmp['timestamp'].iloc[anomalies_index],
    y=tmp['values'].iloc[anomalies_index],
    mode='markers',
    marker=dict(color='red', size=5),
    name='Anomalies'
))
fig = FigureWidgetResampler(go.Figure())
fig.show()

Number of anomalies detected: 131400
Indices of anomalies: [    324     325     326 ... 2626952 2626953 2626954]


In [24]:
anomalies_index

array([    327,     328,     329, ..., 2634155, 2634156, 2634157],
      dtype=int64)

In [ ]:

fig.add_trace(go.Scatter(
    x=tmp['timestamp'],
    y=tmp['values'],
    mode='lines',
    name='Sin Curve'
))

fig.update_layout(
    title='Sin Curve',
    xaxis_title='Timestamp',
    yaxis_title='Values'
)

display(fig)

# fig.show()